In [1]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ========= paths =========
train_path = "/home/sz4544/EEG-motor-imagery-main/project/train1800_raw_EEG.h5"
model_path = "/home/sz4544/EEG-motor-imagery-main/project/models/diffusion_train1800_v2.pt"
scaler_path = "/home/sz4544/EEG-motor-imagery-main/project/models/diffusion_train1800_v2_scaler.npz"
save_aug_path = "/home/sz4544/EEG-motor-imagery-main/project/train1800_diffusion_aug_v2.h5"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# ========= load raw train =========
f = h5py.File(train_path, "r")
x_train = f["data"][:].astype(np.float32)
y_train = f["tasks"][:]
f.close()

print("raw train shape:", x_train.shape, y_train.shape)
print("raw labels:", np.unique(y_train, return_counts=True))

# keep raw copy for final saving
x_train_raw = x_train.copy()

# label remap to 0,1,2,3
y_train = y_train.astype(np.int64)
if np.array_equal(np.unique(y_train), np.array([1, 2, 3, 4])):
    y_train = y_train - 1

print("remapped labels:", np.unique(y_train, return_counts=True))

# ========= load scaler =========
scaler = np.load(scaler_path)
mean = scaler["mean"]   # shape (1,1,64)
std = scaler["std"]     # shape (1,1,64)

# standardize train for diffusion domain
x_train = (x_train - mean) / std
print("standardized train mean/std:", x_train.mean(), x_train.std())

# reshape for diffusion input: (N,64,640)
x_train = np.transpose(x_train, (0, 2, 1))
print("diffusion-domain x_train shape:", x_train.shape)

# ===== check real sample statistics in diffusion domain =====
real_means = np.array([np.mean(x_train[i]) for i in range(len(x_train))])
real_stds = np.array([np.std(x_train[i]) for i in range(len(x_train))])
real_maxabs = np.array([np.max(np.abs(x_train[i])) for i in range(len(x_train))])

print("REAL diffusion-domain sample stats:")
print("mean percentiles:", np.percentile(real_means, [0, 1, 5, 25, 50, 75, 95, 99, 100]))
print("std percentiles:", np.percentile(real_stds, [0, 1, 5, 25, 50, 75, 95, 99, 100]))
print("maxabs percentiles:", np.percentile(real_maxabs, [0, 1, 5, 25, 50, 75, 95, 99, 100]))

# ========= diffusion model definition =========
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=1)
        return emb

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, class_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.class_mlp = nn.Linear(class_dim, out_ch)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.bn2 = nn.BatchNorm1d(out_ch)

    def forward(self, x, t_emb, c_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb).unsqueeze(-1)
        h = h + self.class_mlp(c_emb).unsqueeze(-1)
        h = F.silu(self.bn1(h))
        h = self.conv2(h)
        h = F.silu(self.bn2(h))
        return h

class SimpleCondUNet1D(nn.Module):
    def __init__(self, n_channels=64, n_classes=4, base=64, time_dim=128, class_dim=64):
        super().__init__()
        self.time_emb = SinusoidalPosEmb(time_dim)
        self.class_emb = nn.Embedding(n_classes, class_dim)

        self.in_proj = nn.Conv1d(n_channels, base, 3, padding=1)
        self.block1 = Block(base, base, time_dim, class_dim)
        self.down1 = nn.Conv1d(base, base * 2, 4, stride=2, padding=1)
        self.block2 = Block(base * 2, base * 2, time_dim, class_dim)
        self.down2 = nn.Conv1d(base * 2, base * 4, 4, stride=2, padding=1)
        self.block3 = Block(base * 4, base * 4, time_dim, class_dim)

        self.up1 = nn.ConvTranspose1d(base * 4, base * 2, 4, stride=2, padding=1)
        self.block4 = Block(base * 4, base * 2, time_dim, class_dim)
        self.up2 = nn.ConvTranspose1d(base * 2, base, 4, stride=2, padding=1)
        self.block5 = Block(base * 2, base, time_dim, class_dim)

        self.out_proj = nn.Conv1d(base, n_channels, 3, padding=1)

    def forward(self, x, t, y):
        t_emb = self.time_emb(t)
        c_emb = self.class_emb(y)

        x0 = self.in_proj(x)
        x1 = self.block1(x0, t_emb, c_emb)
        x2 = self.down1(x1)
        x2 = self.block2(x2, t_emb, c_emb)
        x3 = self.down2(x2)
        x3 = self.block3(x3, t_emb, c_emb)

        x = self.up1(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.block4(x, t_emb, c_emb)

        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.block5(x, t_emb, c_emb)

        return self.out_proj(x)

# ========= load model =========
model = SimpleCondUNet1D().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("model loaded")
T = 1000
def sample_one(class_id, clip_value=3.0):
    x = torch.randn(1, 64, 640, device=device)
    y = torch.tensor([class_id], device=device, dtype=torch.long)

    for t in reversed(range(T)):
        t_tensor = torch.tensor([t], device=device, dtype=torch.float32)

        with torch.no_grad():
            eps = model(x, t_tensor, y)

        alpha_t = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t = betas[t]

        x = (x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps) / torch.sqrt(alpha_t)

        if t > 0:
            noise = torch.randn_like(x)
            x = x + torch.sqrt(posterior_variance[t]) * noise

        x = torch.clamp(x, -clip_value, clip_value)

    return x.detach().cpu().numpy()[0]
# ===== quick check: generate a few samples WITHOUT filtering =====


T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
posterior_variance = torch.clamp(posterior_variance, min=1e-12)

for c in range(4):
    print("\nFAKE quick stats for class", c)
    for k in range(5):
        s = sample_one(c)
        print(
            "sample", k,
            "mean =", float(np.mean(s)),
            "std =", float(np.std(s)),
            "maxabs =", float(np.max(np.abs(s)))
        )

# ========= safer diffusion schedule =========
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
posterior_variance = torch.clamp(posterior_variance, min=1e-12)



def fake_is_valid(fake_sample, real_std_min=0.4, real_std_max=1.2, max_abs_limit=4.0):
    sample_std = np.std(fake_sample)
    sample_mean = np.mean(fake_sample)
    sample_max_abs = np.max(np.abs(fake_sample))

    if not np.isfinite(sample_std):
        return False
    if not np.isfinite(sample_mean):
        return False
    if not np.isfinite(sample_max_abs):
        return False

    if sample_std < real_std_min or sample_std > real_std_max:
        return False

    if sample_max_abs > max_abs_limit:
        return False

    return True

# ========= generate only a small number of valid fakes =========
target_per_class = 20
max_trials_per_class = 300

fake_list = []
fake_label = []

for c in range(4):
    print("class", c)
    kept = 0
    tried = 0

    while kept < target_per_class and tried < max_trials_per_class:
        tried += 1

        if tried % 20 == 0:
            print("class", c, "tried", tried, "kept", kept)

        x = sample_one(c)

        if fake_is_valid(x):
            fake_list.append(x)
            fake_label.append(c)
            kept += 1

    print("class", c, "final kept =", kept, "out of tried =", tried)

fake_x = np.stack(fake_list)
fake_y = np.array(fake_label)

print("fake_x:", fake_x.shape)
print("fake_y:", fake_y.shape)
print("fake_y counts:", np.unique(fake_y, return_counts=True))

cuda
raw train shape: (1800, 640, 64) (1800,)
raw labels: (array([1., 2., 3., 4.]), array([450, 450, 450, 450]))
remapped labels: (array([0, 1, 2, 3]), array([450, 450, 450, 450]))
standardized train mean/std: -2.8500283e-06 0.9842581
diffusion-domain x_train shape: (1800, 64, 640)
REAL diffusion-domain sample stats:
mean percentiles: [-2.44724512 -0.86997034 -0.33465129 -0.07803437 -0.00626281  0.06138756
  0.31676409  1.199641    1.81034207]
std percentiles: [0.25688761 0.28492849 0.32681829 0.49457722 0.66828385 0.89903714
 1.73293952 2.93165534 3.97933555]
maxabs percentiles: [ 1.14877844  1.60279977  2.08257633  2.9498052   3.83619595  5.70871043
  8.44314337 10.16624984 15.86707115]
model loaded

FAKE quick stats for class 0
sample 0 mean = 0.013621948659420013 std = 2.386601686477661 maxabs = 3.0
sample 1 mean = 0.025822019204497337 std = 2.3924319744110107 maxabs = 3.0
sample 2 mean = -0.004345295950770378 std = 2.3834428787231445 maxabs = 3.0
sample 3 mean = -0.011579053476452

KeyboardInterrupt: 

In [2]:
def mix_is_valid(mix_sample, real_std_min=0.4, real_std_max=1.2, max_abs_limit=4.0):
    sample_std = np.std(mix_sample)
    sample_mean = np.mean(mix_sample)
    sample_max_abs = np.max(np.abs(mix_sample))

    if not np.isfinite(sample_std):
        return False
    if not np.isfinite(sample_mean):
        return False
    if not np.isfinite(sample_max_abs):
        return False

    if sample_std < real_std_min or sample_std > real_std_max:
        return False

    if sample_max_abs > max_abs_limit:
        return False

    return True


aug_list = []
aug_label = []

for i in range(len(fake_x)):
    c = fake_y[i]

    idx = np.where(y_train == c)[0]
    if len(idx) == 0:
        continue

    j = np.random.choice(idx)

    real = x_train[j]   # standardized domain, shape (64,640)
    fake = fake_x[i]    # standardized domain, shape (64,640)

    lam = np.random.uniform(0.9, 0.98)
    mix = lam * real + (1.0 - lam) * fake

    if mix_is_valid(mix):
        aug_list.append(mix)
        aug_label.append(c)

aug_x = np.stack(aug_list)
aug_y = np.array(aug_label)

print("aug_x:", aug_x.shape)
print("aug_y:", aug_y.shape)
print("aug_y counts:", np.unique(aug_y, return_counts=True))

NameError: name 'fake_x' is not defined

In [ ]:
# inverse-transform aug_x back to raw EEG domain
mean_ch = mean.reshape(64, 1)
std_ch = std.reshape(64, 1)

aug_x_raw = aug_x * std_ch[np.newaxis, :, :] + mean_ch[np.newaxis, :, :]

print("aug_x_raw:", aug_x_raw.shape)
print("aug_x_raw sample stats:")
for i in range(min(5, len(aug_x_raw))):
    print(i, np.mean(aug_x_raw[i]), np.std(aug_x_raw[i]), aug_y[i])

# transpose back to raw train shape (N,640,64)
aug_x_raw_for_save = np.transpose(aug_x_raw, (0, 2, 1))

x_all = np.concatenate([x_train_raw, aug_x_raw_for_save], axis=0)
y_all = np.concatenate([y_train, aug_y], axis=0)

print("x_all:", x_all.shape)
print("y_all:", y_all.shape)
print("final label counts:", np.unique(y_all, return_counts=True))

print("first 5 real-like sample stats:")
for i in [0, 1, 2, 100, 500]:
    print(i, np.mean(x_all[i]), np.std(x_all[i]), y_all[i])

start_aug = len(x_train_raw)
print("first 5 aug-like sample stats:")
for i in range(start_aug, min(start_aug + 5, len(x_all))):
    print(i, np.mean(x_all[i]), np.std(x_all[i]), y_all[i])

In [ ]:
save_aug_path = "/home/sz4544/EEG-motor-imagery-main/project/train1800_diffusion_aug_v2.h5"

f = h5py.File(save_aug_path, "w")
f.create_dataset("data", data=x_all.astype(np.float32))
f.create_dataset("tasks", data=y_all.astype(np.int64))
f.close()

print("saved", save_aug_path)